# Atelier TensorFlow / Keras — Prédiction de la consommation énergétique

**Contexte.** Des bâtiments sont équipés de capteurs. Pour chaque observation on connaît la température, l'humidité et le nombre d'occupants. On veut construire un **réseau de neurones** qui prédit la **consommation énergétique** à partir de ces trois caractéristiques.

C'est un problème de **régression** : la cible (consommation) est un nombre continu.

## Partie 0 — Mise en place de l'environnement

On importe les trois librairies demandées :
- **numpy** : calcul numérique et génération des données aléatoires,
- **tensorflow / keras** : construction et entraînement du réseau de neurones,
- **matplotlib** : visualisation.

> Si TensorFlow n'est pas installé : `pip install tensorflow`

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version :", tf.__version__)

TensorFlow version : 2.21.0


On **fixe les graines aléatoires** (numpy et TensorFlow). Cela rend l'exécution **reproductible** : on obtiendra les mêmes données générées et le même entraînement à chaque relance.

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

## Partie 1 — Génération du dataset

On simule **1000 observations**. Pour chaque variable on choisit une loi de probabilité adaptée :

| Variable | Loi | Paramètres |
|---|---|---|
| `temperature` | Normale (gaussienne) | moyenne = 25 °C, écart-type = 4 °C |
| `humidite` | Uniforme | entre 30 % et 80 % |
| `occupants` | Entiers uniformes | de 1 à 49 inclus |

- **loi normale** : les valeurs se concentrent autour de la moyenne (25) et deviennent rares en s'en éloignant.
- **loi uniforme** : toutes les valeurs entre 30 et 80 ont la même chance de sortir.

In [3]:
n = 1000

temperature = np.random.normal(loc=25, scale=4, size=n)      # moyenne 25, ecart-type 4
humidite    = np.random.uniform(low=30, high=80, size=n)     # uniforme entre 30 et 80
occupants   = np.random.randint(low=1, high=50, size=n)      # entiers 1..49 (high exclu)

print("temperature :", temperature[:5])
print("humidite    :", humidite[:5])
print("occupants   :", occupants[:5])

temperature : [26.98685661 24.4469428  27.59075415 31.09211943 24.0633865 ]
humidite    : [38.37412911 35.22839202 61.82151248 65.32378632 31.57930724]
occupants   : [ 8 43 20 33 26]


### Construction de la cible `consommation`

On applique la formule "physique" donnée par l'énoncé :

$$ \text{consommation} = 50 + 5\times\text{temp} + 1.5\times\text{humidite} + 4\times\text{occupants} + \text{bruit} $$

- **50** = consommation de base (0 °C, 0 % humidité, pièce vide),
- **+5** par degré, **+1.5** par % d'humidité, **+4** par occupant,
- le **bruit** (loi normale de moyenne 0, écart-type 10) simule les imprévus : dans la vraie vie aucune formule n'est parfaite. C'est justement ce que le réseau devra apprendre à approcher **sans** connaître la formule.

In [4]:
bruit = np.random.normal(loc=0, scale=10, size=n)

consommation = 50 + 5*temperature + 1.5*humidite + 4*occupants + bruit

print("consommation :", consommation[:5])

consommation : [281.62608854 406.67868707 348.62841819 433.63621523 318.9414931 ]


### Matrice des caractéristiques `X` et cible `y`

- `X` regroupe les 3 variables explicatives → forme **(1000, 3)**.
- `y` contient la consommation → forme **(1000,)**.

On convertit en **float32** : c'est le format optimisé pour les calculs sur GPU/CPU dans TensorFlow.

In [5]:
X = np.column_stack([temperature, humidite, occupants]).astype("float32")
y = consommation.astype("float32")

print("Forme de X :", X.shape)   # (1000, 3)
print("Forme de y :", y.shape)   # (1000,)
print("\n5 premieres lignes de X :\n", X[:5])

Forme de X : (1000, 3)
Forme de y : (1000,)

5 premieres lignes de X :
 [[26.986856 38.37413   8.      ]
 [24.446943 35.228394 43.      ]
 [27.590754 61.821514 20.      ]
 [31.09212  65.323784 33.      ]
 [24.063387 31.579308 26.      ]]


## Partie 2 — Découpage Train / Test

On sépare les données en deux :
- **train (80 %)** : sert à *entraîner* le modèle,
- **test (20 %)** : sert à *évaluer* le modèle sur des données jamais vues.

`random_state=42` garantit la **reproductibilité** du découpage.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train :", X_train.shape, y_train.shape)
print("Test  :", X_test.shape,  y_test.shape)

Train : (800, 3) (800,)
Test  : (200, 3) (200,)
